# Q9: Data Cleaning and Preparation

**Problem Statement:** Analyzing Customer Churn in a Telecommunications Company

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

customers = pd.read_csv("Telecom_Customer_Churn.csv")
print("Original shape:", customers.shape)
print(customers.head().to_string(index=False))
print("Missing values:\n", customers.isna().sum().to_string())


Original shape: (241, 8)
CustomerID Gender  SeniorCitizen  Tenure       Contract  MonthlyCharges  TotalCharges Churn
    TC0001   Male              0      38 Month-to-month           53.62       2113.97    No
    TC0002 Female              0      39 Month-to-month           46.31       1818.96   Yes
    TC0003 Female              1      21 Month-to-month           72.14       1565.46   Yes
    TC0004   Male              0      30 Month-to-month          100.92       3172.79    No
    TC0005   Male              1      42       One year           95.11       4009.03    No
Missing values:
 CustomerID        0
Gender            0
SeniorCitizen     0
Tenure            0
Contract          0
MonthlyCharges    1
TotalCharges      1
Churn             0


In [2]:
# Remove duplicate customers and standardize inconsistent categorical values.
customers = customers.drop_duplicates(subset="CustomerID").copy()
customers["Gender"] = customers["Gender"].astype("string").str.strip().str.title()
customers["Contract"] = customers["Contract"].astype("string").str.strip().str.title()
customers["Churn"] = customers["Churn"].astype("string").str.strip().str.title()
for column in ["Tenure", "MonthlyCharges", "TotalCharges"]:
    customers[column] = pd.to_numeric(customers[column], errors="coerce")
    customers[column] = customers[column].fillna(customers[column].median())
for column in ["Gender", "Contract"]:
    customers[column] = customers[column].fillna(customers[column].mode().iloc[0])
print("After cleaning:", customers.shape)
print("Remaining missing values:", int(customers.isna().sum().sum()))


After cleaning: (240, 8)
Remaining missing values: 0


In [3]:
# Cap extreme numeric values using IQR bounds.
for column in ["MonthlyCharges", "TotalCharges"]:
    q1, q3 = customers[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    customers[column] = customers[column].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
customers["AverageCharge"] = customers["TotalCharges"] / customers["Tenure"].clip(lower=1)
customers["TenureGroup"] = pd.cut(customers["Tenure"], bins=[0, 12, 24, 48, 100], labels=["New", "Growing", "Established", "Long-term"])
print(customers[["Tenure", "TenureGroup", "AverageCharge", "Churn"]].head().to_string(index=False))


 Tenure TenureGroup  AverageCharge Churn
     38 Established      55.630789    No
     39 Established      46.640000   Yes
     21     Growing      74.545714   Yes
     30 Established     105.759667    No
     42 Established      95.453095    No


In [4]:
# Export the cleaned records before encoding or scaling them for modeling.
customers.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)
X = pd.get_dummies(customers.drop(columns=["CustomerID", "Churn"]), drop_first=True, dtype=int)
y = customers["Churn"].eq("Yes").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
numeric_columns = ["Tenure", "MonthlyCharges", "TotalCharges", "AverageCharge"]
scaler = StandardScaler()
X_train[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])
X_test[numeric_columns] = scaler.transform(X_test[numeric_columns])
print("Train/test rows:", len(X_train), len(X_test))
print("Churn rate in train/test:", round(y_train.mean(), 3), round(y_test.mean(), 3))
print("Saved Telecom_Customer_Churn_Cleaned.csv")


Train/test rows: 192 48
Churn rate in train/test: 0.286 0.292
Saved Telecom_Customer_Churn_Cleaned.csv
